# DHL API Interaction Investigation

This notebook demonstrates how to interact with the DHL tracking API to fetch and display the current status of a shipment.

## 1. Import Required Libraries

Import the necessary libraries for making HTTP requests and handling JSON responses.

In [1]:
# Import Required Libraries
import requests
import json

## 2. Define Constants and Configuration

Set up the DHL search URL, user agent, and default parameters.

In [2]:
# Define DHL API constants and configuration
SEARCH_URL = "https://www.dhl.de/int-verfolgen/data/search"
USER_AGENT = (
    "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/136.0.0.0 Safari/537.36"
)
DEFAULT_LANGUAGE = "de"
DEFAULT_DOMAIN = "de"

## 3. Fetch Tracking Data

Implement and call the `fetch_tracking_data` function using a sample piececode to retrieve raw JSON tracking data from the DHL API.

In [3]:
def fetch_tracking_data(piececode: str, language: str = DEFAULT_LANGUAGE, domain: str = DEFAULT_DOMAIN) -> dict:
    """
    Fetch tracking data from DHL API for a given piececode.
    """
    response = requests.get(
        SEARCH_URL,
        params={
            "piececode": piececode,
            "noRedirect": "true",
            "language": language,
            "lang": language,
            "domain": domain,
        },
        headers={
            "accept": "application/json",
            "user-agent": USER_AGENT,
        },
        timeout=30,
    )
    response.raise_for_status()
    return response.json()

# Example: Use a sample (invalid) piececode for demonstration
sample_piececode = "123456789012"
try:
    tracking_payload = fetch_tracking_data(sample_piececode)
    print(json.dumps(tracking_payload, indent=2, ensure_ascii=False))
except Exception as e:
    print(f"Error fetching tracking data: {e}")

{
  "sendungen": [
    {
      "id": "123456789012",
      "hasCompleteDetails": true,
      "sendungsinfo": {
        "gesuchteSendungsnummer": "123456789012",
        "sendungsrichtung": "ANKOMMEND"
      },
      "sendungsdetails": {
        "sendungsnummern": {
          "sendungsnummer": "123456789012"
        },
        "sendungsverlauf": {
          "fortschritt": 0,
          "maximalFortschritt": 5,
          "farbe": 0,
          "events": []
        },
        "istZugestellt": false,
        "ruecksendung": false,
        "retoure": false,
        "expressSendung": false,
        "quelle": "PAKET",
        "unplausibel": false,
        "invalidTimeOfDay": false,
        "isShipperPlz": false,
        "showQualityLevelHint": false,
        "twoManHandling": false
      },
      "versandDatumBenoetigt": false,
      "nichtAnzeigen": true,
      "sendungNichtGefunden": {
        "keineDatenVerfuegbar": true,
        "keineDhlPaketSendung": true
      },
      "paeckchen": false

## 4. Extract and Display Current Status

Implement the `extract_current_status` function to parse the shipment response and extract the current status.

In [5]:
tracking_payload

{'sendungen': [{'id': '123456789012',
   'hasCompleteDetails': True,
   'sendungsinfo': {'gesuchteSendungsnummer': '123456789012',
    'sendungsrichtung': 'ANKOMMEND'},
   'sendungsdetails': {'sendungsnummern': {'sendungsnummer': '123456789012'},
    'sendungsverlauf': {'fortschritt': 0,
     'maximalFortschritt': 5,
     'farbe': 0,
     'events': []},
    'istZugestellt': False,
    'ruecksendung': False,
    'retoure': False,
    'expressSendung': False,
    'quelle': 'PAKET',
    'unplausibel': False,
    'invalidTimeOfDay': False,
    'isShipperPlz': False,
    'showQualityLevelHint': False,
    'twoManHandling': False},
   'versandDatumBenoetigt': False,
   'nichtAnzeigen': True,
   'sendungNichtGefunden': {'keineDatenVerfuegbar': True,
    'keineDhlPaketSendung': True},
   'paeckchen': False}],
 'mergedAnonymousShipmentListIds': [],
 'rateLimited': False}

In [4]:
def extract_current_status(payload: dict) -> str:
    """
    Extract the current status from the DHL tracking payload.
    """
    shipments = payload.get("sendungen")
    if not shipments:
        raise ValueError("No shipments found in DHL response")
    return shipments[0]["sendungsdetails"]["sendungsverlauf"]["status"]

# Try extracting the status from the previous payload (if available)
try:
    status = extract_current_status(tracking_payload)
    print(f"Current status: {status}")
except Exception as e:
    print(f"Error extracting status: {e}")

Error extracting status: 'status'


## 5. Handle Errors and Edge Cases

Demonstrate error handling for invalid piececodes, missing shipments, and HTTP errors.

In [ ]:
# Demonstrate error handling with various scenarios
invalid_piececode = "INVALIDCODE"
try:
    payload = fetch_tracking_data(invalid_piececode)
    status = extract_current_status(payload)
    print(f"Status for invalid piececode: {status}")
except requests.HTTPError as http_err:
    print(f"HTTP error occurred: {http_err}")
except ValueError as val_err:
    print(f"Value error: {val_err}")
except Exception as e:
    print(f"Unexpected error: {e}")